In [ ]:
from IPython.display import clear_output

%pip install kagglehub catboost lightgbm tqdm -q

clear_output()

import pandas as pd
import numpy as np
import sklearn.preprocessing
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm
%matplotlib inline


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# 1. What does our target variable (charges) look like?
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:

print(f"Before: {df.shape}")
df = df.drop(columns ='Order_ID')
print(f"After dropping: {df.shape}")

In [ ]:
# Task 2: Write your code here:
missing_values = df.isnull().sum()
print("Missing Values per Column:")
print(missing_values[missing_values > 0])
df.head(1)

for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    df[col] = df[col].fillna('unknown')

for col in ['Courier_Experience_yrs' , 'Delivery_Time' ] :
  df[col] = df[col].fillna(df[col].mean())

print("Missing values remaining:", df.isnull().sum().sum())

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True) # Without inplace=True: Pandas creates a copy of your data with the duplicates removed, but your original df stays messy.
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)
duplicates = df.duplicated().sum()
print(f"Number of Duplicate Samples after the procees done: {duplicates}")

In [ ]:
# Task 4: Write your code here:

categorical_cols = df.select_dtypes(include=["object"]).columns

for col in categorical_cols:
  onehot_encoder = OneHotEncoder(sparse_output=False)
  df[col] = onehot_encoder.fit_transform(df[categorical_cols].astype(str))

df.head()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler #import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 6: Write your code here:
# 1. Is the target imbalanced?
def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  df[target_column].hist()  # Yeah you can just do this :)
  plt.show()

check_target_imbalance(df, "Delivery_Time")
# if data imbalence f score and stratifiesd
# tearget balanced :) it means it is not omlabnced

In [ ]:
# Task 1: Write your code here:
X = df.drop(columns="Delivery_Time")
y = df['Delivery_Time']


In [ ]:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import  mean_absolute_error


In [ ]:
# Task 2,3,4,5: Write your code here:
n_splits = 5
kf = KFold(n_splits=5, shuffle=True, random_state=42)
model = RandomForestRegressor(n_estimators=200)
mseList = []
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for i in range(5):
    # Train
    model.fit(X_train, y_train)
    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mse = mean_absolute_error(y_test, y_pred)
    mseList.append(mse)

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': X,
    'importance': model.feature_importances_ # so special and confined for spicific models one of them is linear regression
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
# Transmission distribution
Delivery_Time = df['Delivery_Time'].value_counts()
plt.figure(figsize=(10, 5))
plt.bar(Delivery_Time.index, Delivery_Time.values, color='teal')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery_Time Type')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()


In [ ]:
# Task Bonus: Write your code here: